# DU Timing Window Auto-Tuning Tool
This notebook analyzes RU logs and adjusts DU FH timing parameters dynamically based on defined timing windows.

In [ ]:
import re

# Timing window key parameters (in nanoseconds)
timing_params = {
    "Tadv_cp_dl": 125,
    "T2a_cp_dl": (259, 470),
    "T2a_cp_ul": (125, 1200),
    "T2a_up": (70, 345),
    "Ta3": (50, 171),
    "T1a_cp_dl": (258, 392),
    "T1a_cp_ul": (285, 300),
    "T1a_up": (155, 300),
    "Ta4": (0, 200)
}


In [ ]:
# Load RU log (example from ap3_boot.log)
with open("ap3_boot.log", "r") as f:
    ru_log = f.read()

# Extract key timing from log
import numpy as np

dpd_delay = None
tx_dump = None
rx_dump = None

for line in ru_log.splitlines():
    if "DPD path delay" in line:
        dpd_delay = float(re.search(r"= ([\d\.]+)", line).group(1))
    elif "TX dump delay" in line:
        tx_dump = int(re.search(r"= (\d+)", line).group(1))
    elif "RX dump delay" in line:
        rx_dump = int(re.search(r"= (\d+)", line).group(1))

print(f"DPD delay: {dpd_delay} samples")
print(f"TX dump delay: {tx_dump}, RX dump delay: {rx_dump}")


In [ ]:
# Validate timing values
violations = []

if dpd_delay is not None:
    dpd_ns = dpd_delay * (1000 / 30.72)  # 30.72 MHz => ~32.55 ns/sample
    if not (timing_params["T2a_up"][0] <= dpd_ns <= timing_params["T2a_up"][1]):
        violations.append(f"T2a_up violated: {dpd_ns:.2f} ns not in {timing_params['T2a_up']}")

if tx_dump and rx_dump:
    tx_rx_diff = (tx_dump - rx_dump) * (1000 / 30.72)
    if not (timing_params["Ta4"][0] <= tx_rx_diff <= timing_params["Ta4"][1]):
        violations.append(f"Ta4 violated: {tx_rx_diff:.2f} ns not in {timing_params['Ta4']}")

print("Violations found:" if violations else "All timing within range.")
for v in violations:
    print(" -", v)


In [ ]:
# Example rule-based adjustment
suggestions = []

if tx_rx_diff and tx_rx_diff > timing_params["Ta4"][1]:
    suggestions.append("Consider increasing sl_ahead by 1 to reduce RX lateness.")

if dpd_ns and dpd_ns > timing_params["T2a_up"][1]:
    suggestions.append("Consider increasing tx_amp_backoff_dB or reducing buffering in RU.")

print("Suggested adjustments:")
for s in suggestions:
    print(" -", s)
